In [16]:
import os 
import numpy as np 
import pandas as pd 
import nest_asyncio 
import httpx
import phoenix as px 
from opentelemetry import trace #or ->from phoenix.trace import tracer #if using Phoenix helper
from phoenix.otel import register 
from openinference.instrumentation.langchain import LangChainInstrumentor
from langchain_groq import ChatGroq 
from langchain_core.prompts import ChatPromptTemplate

In [37]:

from dotenv import load_dotenv
load_dotenv()

True

In [5]:
tracer_provider=register()
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: default
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [17]:
tracer=trace.get_tracer(__name__)

In [6]:
df = pd.read_parquet(
    "http://storage.googleapis.com/arize-phoenix-assets/datasets/"
    "unstructured/llm/context-retrieval/langchain/database.parquet"
)

df.head()

,text,text_vector
0,\nAccess tutorials of what's possible with Ari...,"[-0.0036313518733896957, 0.0026182831234325553..."
1,"\n{% hint style=""info"" %}\nYour model type det...","[-0.006279695251247202, -0.0005314257337369415..."
2,\nExamples for logging explainability metrics....,"[-0.007306394403318994, 0.0007199545863166625,..."
3,\n| Google Cloud Services | Link |\n| ------...,"[0.002060703379662884, -0.0037287262540240764,..."
4,\n| Sending 10 Million Inferences to Arize in ...,"[-0.008325811390756237, -0.008001045242455413,..."


In [8]:
llm=ChatGroq(
    api_key=os.environ['GROQ_API_KEY'],
    model_name='llama-3.1-8b-instant',
    temperature=0
)

In [13]:
prompt=ChatPromptTemplate.from_template(
    "Answer the question:{question}"
)
chain=prompt|llm 

chain.invoke({'question':"what is ml?"})

Transient error StatusCode.UNAVAILABLE encountered while exporting traces to localhost:4317, retrying in 0.96s.
Failed to export traces to localhost:4317, error code: StatusCode.UNAVAILABLE
Transient error StatusCode.UNAVAILABLE encountered while exporting traces to localhost:4317, retrying in 0.96s.
Failed to export traces to localhost:4317, error code: StatusCode.UNAVAILABLE
Transient error StatusCode.UNAVAILABLE encountered while exporting traces to localhost:4317, retrying in 0.87s.
Failed to export traces to localhost:4317, error code: StatusCode.UNAVAILABLE


AIMessage(content='"ML" can refer to several things depending on the context. Here are a few possible meanings:\n\n1. **Machine Learning**: In the field of computer science, ML stands for Machine Learning. It is a subset of artificial intelligence (AI) that involves the use of algorithms and statistical models to enable machines to learn from data, make decisions, and improve their performance over time.\n\n2. **Medical Laboratory**: In the medical field, ML stands for Medical Laboratory. It refers to a laboratory that conducts tests and analyzes samples to diagnose and monitor diseases.\n\n3. **Master of Laws**: In education, ML stands for Master of Laws. It is a postgraduate degree in law that is often pursued by individuals who want to specialize in a particular area of law or prepare for a career in law.\n\n4. **Media Literacy**: In education and communication, ML stands for Media Literacy. It refers to the ability to critically evaluate and understand the messages and information 

In [14]:
print(chain.invoke({'question':'what is deep learning'})['content'])

Transient error StatusCode.UNAVAILABLE encountered while exporting traces to localhost:4317, retrying in 1.09s.
Failed to export traces to localhost:4317, error code: StatusCode.UNAVAILABLE
Transient error StatusCode.UNAVAILABLE encountered while exporting traces to localhost:4317, retrying in 0.99s.
Failed to export traces to localhost:4317, error code: StatusCode.UNAVAILABLE
Transient error StatusCode.UNAVAILABLE encountered while exporting traces to localhost:4317, retrying in 0.96s.
Failed to export traces to localhost:4317, error code: StatusCode.UNAVAILABLE


content='**Deep Learning: A Subfield of Machine Learning**\n\nDeep learning is a subfield of machine learning that focuses on the development of artificial neural networks with multiple layers, inspired by the structure and function of the human brain. These neural networks are capable of learning complex patterns in data, such as images, speech, and text, and can be used for a wide range of applications, including:\n\n1. **Image recognition**: Identifying objects, scenes, and activities in images and videos.\n2. **Speech recognition**: Transcribing spoken language into text.\n3. **Natural language processing**: Understanding and generating human language.\n4. **Predictive modeling**: Making predictions based on historical data, such as forecasting stock prices or predicting customer behavior.\n\n**Key Characteristics of Deep Learning**\n\n1. **Multiple layers**: Deep learning models consist of multiple layers of interconnected nodes or "neurons," which process and transform the input 

In [33]:
TAVILY_API_KEY=os.getenv('TAVILY_API_KEY')

In [ ]:


# resp=httpx.post(
#     'https://api.tavily.com/search',
#     headers={
#         "Authorization:f'Bearer {TAVILY_API_KEY}' "
#     },
#     json={
#         'api_key':TAVILY_API_KEY,
#         'max_results':3,
#         'search_depth':'basic',
#         'include_answer':True,
#     }, timeout=8
#     )
# print(resp.status_code)
# data=resp.json() 
# print('data',data) 
# print(data.keys)
# answer=data.get('answer',"") or ""
# print('answer',answer)
# sinppets=[item.get('content','') for item in data.get('results',[])]
# print('sinppets',sinppets)



422
data {'detail': [{'type': 'missing', 'loc': ['body', 'query'], 'msg': 'Field required', 'input': {'api_key': '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', 'max_results': 3, 'search_depth': 'basic', 'include_answer': True, 'raw_input': {'api_key': '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', 'max_results': 3, 'search_depth': 'basic', 'include_answer': True}}}]}
<built-in method keys of dict object at 0x12f16e680>
answer 
sinppets []


In [ ]:
# --- Helper functions for tools ---
@tracer.chain(name="search-api")
def _search_api(query: str) -> str | None:
    api_key = os.getenv("TAVILY_API_KEY")

    resp = httpx.post(
        "https://api.tavily.com/search",
        json={
            "api_key": api_key,
            "query": query,
            "max_results": 3,
            "search_depth": "basic",
            "include_answer": True,
        },
        timeout=8,
    )
    print(f'the json {resp}') 

    data = resp.json()
    print('data',data)
    answer = data.get("answer", "") or ""
    print('answer',answer) 

    snippets = [item.get("content", "") for item in data.get("results", [])]
    print('snippets',snippets)

    combined = " ".join([answer] + snippets).strip()
    return combined[:400] if combined else None


@tracer.chain(name="weather-api")
def _weather(dest): #dest = destination (place name)
    print('dest',dest)
    
    g = httpx.get(f"https://geocoding-api.open-meteo.com/v1/search?name={dest}")
    print('g',g)
    if g.status_code != 200 or not g.json().get("results"):
        return ""

    lat = g.json()["results"][0]["latitude"]
    lon = g.json()["results"][0]["longitude"]
    print(lat)
    print(lon) 
    

    w = httpx.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
    ).json()

    cw = w.get("current_weather", {})
    return f"Weather now: {cw.get('temperature')}°C, wind {cw.get('windspeed')} km/h."

In [95]:
print(_search_api("latest AI news"))


the json <Response [200 OK]>
data {'query': 'latest AI news', 'follow_up_questions': None, 'answer': 'Recent AI advancements include significant job cuts by Atlassian and Meta unveiling new AI chips. These developments highlight ongoing shifts in the tech industry towards AI-driven innovations. For the latest updates, visit reputable tech news sources.', 'images': [], 'results': [{'url': 'https://www.wsj.com/tech/ai?gaa_at=eafs&gaa_n=AWEtsqcAuFdrsMNVuAGmJMa6coO9_oNrTvhL9EXnJRvP2w__UVywoTLUw6D4&gaa_ts=69bade37&gaa_sig=omJjPQ16W3BifyOyU_R7oAAz8u85Vsby1NmT34ASfttuxz6oKK_s9-1AQAcdnnMa7M0avU_ZKAQ_qziW8RjLeA%3D%3D', 'title': 'Artificial Intelligence - Latest AI News and Analysis - WSJ.com', 'content': 'The latest artificial intelligence news coverage focusing on the technology, tools and the companies building AI technology.', 'score': 0.99986863, 'raw_content': None}, {'url': 'https://aimagazine.com/', 'title': 'AI Magazine: The Latest AI News, Insights and Reports', 'content': 'The No.1 Ma

In [99]:
_weather("Akola")

'Weather now: 26.0°C, wind 7.0 km/h.'

In [98]:

print(_weather("Akola"))

Weather now: 26.0°C, wind 7.0 km/h.


In [68]:
# !pip install agno

In [69]:
from agno.tools import tool

@tool 
def esssential_info(destination:str):
    """get the essential info using Search and weather APIs"""
    parts=[]

    q=f'{destination} travel essentials weather besttime top attraction etiquette'
    s=_search_api(q)
    if s:
        parts.append(f'{destination} essentials:{s}')
    else:
        parts.append(f"{destination} is a popular travel destiotion .Expect local cultrure,cuisine,and landmarks worth exploring")

    weather=_weather(dest=destination)

    if weather:
        parts.append(weather)

    return f'{destination} essentials:\n'+'\n'.join(parts)

In [72]:
@tracer.chain(name="weather-api")
def budget_basics(destination: str, duration: str) -> str:
    """Summarize travel cost categories."""
    q = f"{destination} travel budget average daily costs {duration}"
    s = _search_api(q)
    if s:
        return f"{destination} budget ({duration}): {s}"
    return f"Budget for {duration} in {destination} depends on lodging, meals, transport, and attractions."


@tool
def local_flavor(destination: str, interests: str = "local culture") -> str:
    """Suggest authentic local experiences."""
    q = f"{destination} authentic local experiences {interests}"
    s = _search_api(q)
    if s:
        return f"{destination} {interests}: {s}"
    return f"Explore {destination}'s unique {interests} through markets, neighborhoods, and local eateries."

In [78]:
from agno.agent import Agent 
from agno.models.groq import Groq
import os 

model=Groq(
    id='llama-3.1-8b-instant',
    api_key=os.environ['GROQ_API_KEY']
)

In [79]:
trip_agent=Agent(
    name='TripPlanner',
    role='AI Travel Assistant',
    model=model,
    instructions=("You are a friendly and knowledgeable travel planner. "
        "Combine multiple tools to create a trip plan including essentials, budget, and local flavor. "
        "Keep the tone natural, clear, and under 1000 words."
    ),
    markdown=True ,
    tools=[esssential_info,budget_basics,local_flavor]
)

In [84]:
from phoenix.client import Client

client = Client()

In [85]:
# --- Example usage ---
destination = "Tokyo"
duration = "5 days"
interests = "food, culture"

query = f"""
Plan a {duration} trip to {destination}.
Focus on {interests}.
Include essential info, budget breakdown, and local experiences.
"""


def agent_task(query):
    trip_agent.print_response(query, stream=True)


agent_task(query)

the json <Response [200 OK]>

data {'query': 'Tokyo travel essentials weather besttime top attraction etiquette', 'response_time': 3.73, 
'follow_up_questions': None, 'answer': 'Best time to visit Tokyo is October to November; top attraction is the 
Tokyo Tower; basic etiquette includes bowing and removing shoes indoors.', 'images': [], 'results': [{'title': 
'Weather in Tokyo, Japan', 'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'Tokyo', 
'region': 'Tokyo', 'country': 'Japan', 'lat': 35.6895, 'lon': 139.6917, 'tz_id': 'Asia/Tokyo', 'localtime_epoch': 
1773843055, 'localtime': '2026-03-18 23:10'}, 'current': {'last_updated_epoch': 1773842400, 'last_updated': 
'2026-03-18 23:00', 'temp_c': 15.2, 'temp_f': 59.4, 'is_day': 0, 'condition': {'text': 'Partly cloudy', 'icon': 
'//cdn.weatherapi.com/weather/64x64/night/116.png', 'code': 1003}, 'wind_mph': 11.2, 'wind_kph': 18.0, 
'wind_degree': 220, 'wind_dir': 'SW', 'pressure_mb': 1012.0, 'pressure_in': 29.88, 'precip_mm': 0.0, 'precip_in': 
0.0, 'humidity': 82, 'cloud': 75, 'feelslike_c': 15.2, 'feelslike_f': 59.4, 'windchill_c': 12.1, 'windchill_f': 
53.7, 'heatindex_c': 13.6, 'heatindex_f': 56.5, 'dewpoint_c': 5.8, 'dewpoint_f': 42.5, 'vis_km': 10.0, 'vis_miles':
6.0, 'uv': 0.0, 'gust_mph': 15.7, 'gust_kph': 25.3}}", 'score': 0.8417516, 'raw_content': None}, {'url': 
'https://www.machupicchu.org/best-time-visit-tokyo-2026-seasonal-planning-guide.htm', 'title': 'Best Time Visit 
Tokyo 2026: Seasonal Planning Guide - Machu Picchu', 'content': 'October and November represent Tokyo’s optimal 
months combining comfortable temperatures 15-22°C (59-72°F), minimal rainfall averaging just 4-5 days monthly, 
spectacular autumn foliage displays, moderate tourist volumes, and hotel rates 20-30% below spring peak pricing. 
September suits visitors seeking autumn conditions without November foliage crowds, comfortable temperatures for 
extensive sightseeing, and good value pricing 20-30% below spring cherry blossom season. Hotel rates fluctuate 
40-70% across Tokyo’s seasons with winter (January-February) providing lowest rates at ¥8,000-15,000 ($56-105) for 
business hotels, spring cherry blossom season (late March-early April) commanding premium rates of ¥15,000-25,000 
($105-175) or higher, summer (June-August) offering moderate pricing ¥10,000-18,000 ($70-126), and autumn 
(September-November) maintaining reasonable rates ¥11,000-19,000 ($77-133) with slight increases during November 
foliage peak. Avoid peak periods including cherry blossom season (March 25-April 10), Golden Week (April 26-May 6),
and year-end holidays (December 29-January 5) unless specifically interested in these cultural moments and 
comfortable with associated crowds and premium pricing.', 'score': 0.7715858, 'raw_content': None}, {'url': 
'https://www.japanhighlights.com/japan/best-time-to-visit', 'title': 'Best & Worst Time to Visit Japan 2026/2027 
(First-Timer Guide)', 'content': "Check out our [10-Day Charming Japan Cherry Blossom 
Tour](https://www.japanhighlights.com/japan/tours/private-cherry-blossom), designed to **bring you to the right 
places at the right time** for the best blossom views, tailor-made just for you. Check [our 11-day early spring 
tour](https://www.japanhighlights.com/japan/tours/early-spring-tour)to see cherry blossoms before the big crowds 
arrive. Therefore, the [best time to visit Tokyo](https://www.japanhighlights.com/japan/tokyo/best-time-to-visit), 
Japan may cover four seasons. Click to customize the visiting time from a [11-Day Traditional Japan 
Tour](https://www.japanhighlights.com/japan/tours/traditional-japan)! *   [Japan Weather in April 2026: Cherry 
Blossom Peak & Travel Tips](https://www.japanhighlights.com/japan/weather-in-april). *   [Japan Weather in May 
2026: Good Time Without Crowds & Heat](https://www.japanhighlights.com/japan/weather-in-may). *   [Japan Weather in
September 2026: How to Enjoy a Great Trip during the Typhoon 
Season](https://www.japanhighlights.com/japan

answer Best time to visit Tokyo is October to November; top attraction is the Tokyo Tower; basic etiquette includes
bowing and removing shoes indoors.

snippets ["{'location': {'name': 'Tokyo', 'region': 'Tokyo', 'country': 'Japan', 'lat': 35.6895, 'lon': 139.6917, 
'tz_id': 'Asia/Tokyo', 'localtime_epoch': 1773843055, 'localtime': '2026-03-18 23:10'}, 'current': 
{'last_updated_epoch': 1773842400, 'last_updated': '2026-03-18 23:00', 'temp_c': 15.2, 'temp_f': 59.4, 'is_day': 0,
'condition': {'text': 'Partly cloudy', 'icon': '//cdn.weatherapi.com/weather/64x64/night/116.png', 'code': 1003}, 
'wind_mph': 11.2, 'wind_kph': 18.0, 'wind_degree': 220, 'wind_dir': 'SW', 'pressure_mb': 1012.0, 'pressure_in': 
29.88, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 82, 'cloud': 75, 'feelslike_c': 15.2, 'feelslike_f': 59.4, 
'windchill_c': 12.1, 'windchill_f': 53.7, 'heatindex_c': 13.6, 'heatindex_f': 56.5, 'dewpoint_c': 5.8, 
'dewpoint_f': 42.5, 'vis_km': 10.0, 'vis_miles': 6.0, 'uv': 0.0, 'gust_mph': 15.7, 'gust_kph': 25.3}}", 'October 
and November represent Tokyo’s optimal months combining comfortable temperatures 15-22°C (59-72°F), minimal 
rainfall averaging just 4-5 days monthly, spectacular autumn foliage displays, moderate tourist volumes, and hotel 
rates 20-30% below spring peak pricing. September suits visitors seeking autumn conditions without November foliage
crowds, comfortable temperatures for extensive sightseeing, and good value pricing 20-30% below spring cherry 
blossom season. Hotel rates fluctuate 40-70% across Tokyo’s seasons with winter (January-February) providing lowest
rates at ¥8,000-15,000 ($56-105) for business hotels, spring cherry blossom season (late March-early April) 
commanding premium rates of ¥15,000-25,000 ($105-175) or higher, summer (June-August) offering moderate pricing 
¥10,000-18,000 ($70-126), and autumn (September-November) maintaining reasonable rates ¥11,000-19,000 ($77-133) 
with slight increases during November foliage peak. Avoid peak periods including cherry blossom season (March 
25-April 10), Golden Week (April 26-May 6), and year-end holidays (December 29-January 5) unless specifically 
interested in these cultural moments and comfortable with associated crowds and premium pricing.', "Check out our 
[10-Day Charming Japan Cherry Blossom Tour](https://www.japanhighlights.com/japan/tours/private-cherry-blossom), 
designed to **bring you to the right places at the right time** for the best blossom views, tailor-made just for 
you. Check [our 11-day early spring tour](https://www.japanhighlights.com/japan/tours/early-spring-tour)to see 
cherry blossoms before the big crowds arrive. Therefore, the [best time to visit 
Tokyo](https://www.japanhighlights.com/japan/tokyo/best-time-to-visit), Japan may cover four seasons. Click to 
customize the visiting time from a [11-Day Traditional Japan 
Tour](https://www.japanhighlights.com/japan/tours/traditional-japan)! *   [Japan Weather in April 2026: Cherry 
Blossom Peak & Travel Tips](https://www.japanhighlights.com/japan/weather-in-april). *   [Japan Weather in May 
2026: Good Time Without Crowds & Heat](https://www.japanhighlights.com/japan/weather-in-may). *   [Japan Weather in
September 2026: How to Enjoy a Great Trip during the Typhoon 
Season](https://www.japanhighlights.com/japan/weather-in-september). [12-Day Classic Japan 
Tour](https://www.japanhighlights.com/japan/tours/classic-japan)[![Image 18: We are recommended by these social 
medias.](https://data.japanhighlights.com/grey.gif)](https://www.japanhighlights.com/japan/tours/private-cherry-blo
ssom). [10-Day Charming Japan Cherry Blossom 
Tour](https://www.japanhighlights.com/japan/tours/private-cherry-blossom)[![Image 19: We are recommended by these 
social 
medias.](https://data.japanhighlights.com/grey.gif)](https://www.japanhighlights.com/japan/tours/traditional-japan)
. [![Image 21: We are recommended by these social 
medias.](https://data.japanhighlights.com/grey.gif)](https://www.japanhighlights.com/japan/how-to-plan-a-family-tri
p-to-japan). [Japan's Cherry Blossoms 2026 - Plan for the Full Bloom 
Now](https://www.japanhighl

the json <Response [200 OK]>

...

snippets ["Here's a realistic daily breakdown to help you budget better Accommodation: $80–$250 per night Food: 
$30–$80 per day (yes, amazing meals", 'Accommodation : minimum of 3000 yen per day (sometimes hostel might provide 
the breakfast) · Food : Around 2–3000 yen (cheap option, not fancy,', '$100 per day for food, transportation and 
small souvenirs would be enough. Accommodation varies since options range from capsule hotels to more']

the json <Response [200 OK]>

data {'query': 'Tokyo authentic local experiences food, culture', 'follow_up_questions': None, 'answer': 'Tokyo 
offers authentic local dining experiences at hidden izakayas and food tours through neighborhoods like Shinjuku and
Shibuya. Enjoy sushi, yakiniku, and street food. Tours include visits to Tsukiji Fish Market.', 'images': [], 
'results': [{'url': 
'https://www.magical-trip.com/media/19-best-tokyo-food-tours-2025-authentic-local-experiences-hidden-gems-guide/', 
'title': '19 Best Tokyo Food Tours 2026: Authentic Local Experiences ...', 'content': '| Tour Overview | Enjoy a 
sushi course, yakiniku (Japanese BBQ), and dessert in Shinjuku at night | Tour local-loved restaurants and hidden 
izakaya in Shibuya with a guide | Savor 13 types of Japanese food at 4 restaurants in Shibuya |. | Features | 
Experience fresh seafood at the historic Tsukiji Fish Market   Walk through Tsukiji with a guide and deeply 
experience the area\'s history and culture | Learn about Tsukiji Fish Market\'s history and food culture through 
your guide\'s detailed explanations  Enjoy food walking tours featuring fresh seafood, wagyu beef, fruits, and more
| Experience the vibrant atmosphere of this world-famous fish market  Taste authentic street food including sushi 
and other local specialties |. This tour visits three Izakaya and bars in Shinjuku, including Kabukicho, known as 
one of Japan\'s premier entertainment districts, and the retro drinking alley "Omoide Yokocho Alley," where you can
enjoy drinks and food. This is a recommended Tokyo Food Tours experience that lets you enjoy the entire Tsukiji 
area.', 'score': 0.9999914, 'raw_content': None}, {'url': 
'https://www.foodtourtokyo.com/post/eat-like-a-local-in-tokyo-10-tips', 'title': 'Eat Like a Local in Tokyo: 10 
Tips for Authentic Japanese Dining', 'content': "# Eat Like a Local in Tokyo: 10 Tips for Authentic Japanese 
Dining. ## Eat Like a Local in Tokyo - Let's Get Started. **Eat Like a Local in Tokyo**\xa0and discover how locals 
really eat. Instead of sticking to tourist-friendly spots, use these 10 insider tips to dive deeper into the city’s
food scene and eat like a true local. Small local restaurants can feel intimidating, especially if there’s no 
English menu or the staff doesn’t speak much English. > **Ready to experience Tokyo like a true local?**  **Japan 
Food Travels** **curates tailor-made itineraries filled with neighborhood restaurants, hidden gems, and authentic 
dining experiences - far from the typical tourist circuit.** **See how JFT can elevate your trip****.**. Eating 
like a local in Tokyo means being a little adventurous, stepping outside the usual tourist zones, and trusting the 
process. Whether it’s counter dining, following the locals, or trying something new, these tips will help you find 
some of the best food Tokyo has to offer.", 'score': 0.99996436, 'raw_content': None}, {'url': 
'https://www.withlocals.com/experiences/japan/tokyo/online/food-tours/', 'title': 'Best Food Tours in Tokyo - 
Withlocals', 'content': '# Food Tours in Tokyo. Take your taste buds to paradise on a Tokyo private food tour! ## 
Top 5 Food Tours in Tokyo. 2.5 hours | Food tours | Tokyo. 2.5 hours | food tours | Tokyo. Join your foodie host 
for a food tour in Chinatown and satisfy your cravings for local food, culture, and city highlights. Enjoy 10 
delicious tastings that range from sweet to savory as well as drinks on a tasty food tour in Yokohama. 2.5 hours | 
food tours | Tokyo. Join a local host expert in food and culture on a tour that will satisfy your cultural cravings
for Sake & Sushi with tastings, history, and highlights. 3 hours | Food tours | Tokyo. Enjoy 10 delicious and 
typical tastings that range from sweet to savory as well as drinks on a tasty food tour in Tokyo. ## All Food Tours
in Tokyo.', 'score': 0.9999269, 'raw_content': None}], 'response_time': 1.52, 'request_id': 
'67ed13d5-bed2-4ac5-9d1d-83e3cc76b444'}

answer Tokyo offers authentic local dining experiences at hidden izakayas and food tours through neighborhoods like
Shinjuku and Shibuya. Enjoy sushi, yakiniku, and street food. Tours include visits to Tsukiji Fish Market.

snippets ['| Tour Overview | Enjoy a sushi course, yakiniku (Japanese BBQ), and dessert in Shinjuku at night | Tour
local-loved restaurants and hidden izakaya in Shibuya with a guide | Savor 13 types of Japanese food at 4 
restaurants in Shibuya |. | Features | Experience fresh seafood at the historic Tsukiji Fish Market   Walk through 
Tsukiji with a guide and deeply experience the area\'s history and culture | Learn about Tsukiji Fish Market\'s 
history and food culture through your guide\'s detailed explanations  Enjoy food walking tours featuring fresh 
seafood, wagyu beef, fruits, and more | Experience the vibrant atmosphere of this world-famous fish market  Taste 
authentic street food including sushi and other local specialties |. This tour visits three Izakaya and bars in 
Shinjuku, including Kabukicho, known as one of Japan\'s premier entertainment districts, and the retro drinking 
alley "Omoide Yokocho Alley," where you can enjoy drinks and food. This is a recommended Tokyo Food Tours 
experience that lets you enjoy the entire Tsukiji area.', "# Eat Like a Local in Tokyo: 10 Tips for Authentic 
Japanese Dining. ## Eat Like a Local in Tokyo - Let's Get Started. **Eat Like a Local in Tokyo**\xa0and discover 
how locals really eat. Instead of sticking to tourist-friendly spots, use these 10 insider tips to dive deeper into
the city’s food scene and eat like a true local. Small local restaurants can feel intimidating, especially if 
there’s no English menu or the staff doesn’t speak much English. > **Ready to experience Tokyo like a true local?**
**Japan Food Travels** **curates tailor-made itineraries filled with neighborhood restaurants, hidden gems, and 
authentic dining experiences - far from the typical tourist circuit.** **See how JFT can elevate your trip****.**. 
Eating like a local in Tokyo means being a little adventurous, stepping outside the usual tourist zones, and 
trusting the process. Whether it’s counter dining, following the locals, or trying something new, these tips will 
help you find some of the best food Tokyo has to offer.", '# Food Tours in Tokyo. Take your taste buds to paradise 
on a Tokyo private food tour! ## Top 5 Food Tours in Tokyo. 2.5 hours | Food tours | Tokyo. 2.5 hours | food tours 
| Tokyo. Join your foodie host for a food tour in Chinatown and satisfy your cravings for local food, culture, and 
city highlights. Enjoy 10 delicious tastings that range from sweet to savory as well as drinks on a tasty food tour
in Yokohama. 2.5 hours | food tours | Tokyo. Join a local host expert in food and culture on a tour that will 
satisfy your cultural cravings for Sake & Sushi with tastings, history, and highlights. 3 hours | Food tours | 
Tokyo. Enjoy 10 delicious and typical tastings that range from sweet to savory as well as drinks on a tasty food 
tour in Tokyo. ## All Food Tours in Tokyo.']

In [59]:
# g=httpx.get(f"https://geocoding-api.open-meteo.com/v1/search?name={dest}")


In [86]:
import pandas as pd

# --- Example queries ---
queries = [
    "Plan a 7-day trip to Italy focused on art, history, and local food. Include essential travel info, a budget estimate, and key attractions in Rome, Florence, and Venice.",
    "Create a 4-day itinerary for Seoul centered on K-pop, fashion districts, and street food. Include transportation tips and a mid-range budget.",
    "Plan a romantic 5-day getaway to Paris with emphasis on museums, wine tasting, and scenic walks. Provide cost estimates and essential travel notes.",
    "Design a 3-day budget trip to Mexico City focusing on food markets, archaeological sites, and nightlife. Include daily cost breakdowns.",
    "Prepare a 6-day itinerary for New Zealand’s South Island with a focus on outdoor adventure, hikes, and photography spots. Include travel logistics and gear essentials.",
    "Plan a 10-day trip across Spain, hitting Barcelona, Madrid, and Seville. Focus on architecture, tapas, and cultural festivals. Include a detailed budget.",
    "Create a 5-day family-friendly itinerary for Singapore with theme parks, nature activities, and kid-friendly dining. Include entry fees and transit costs.",
    "Plan a 4-day luxury spa and relaxation trip to Bali. Include premium resorts, wellness activities, and a high-end budget.",
    "Design a 7-day solo backpacking trip through Thailand with hostels, street food, and cultural attractions. Provide safety essentials and budget breakdown.",
    "Create a 3-day weekend itinerary for New York City focusing on art galleries, rooftop restaurants, and iconic attractions. Include estimated costs.",
]

dataset_df = pd.DataFrame(data={"input": queries})

dataset = client.datasets.create_dataset(
    dataframe=dataset_df, name="travel-questions", input_keys=["input"]
)

In [60]:
queries = [
    "What is AI?",
    "Weather in Mumbai",
    "Latest news about machine learning",
    "release data of LLM"
]

In [87]:
ANSWER_RELEVANCE_PROMPT_TEMPLATE = """
You will be given a travel-planning query and an itinerary answer. Your task is to decide whether
the answer correctly follows the user's instructions. An answer is "incorrect" if it contradicts,
ignores, or fails to include required elements from the query (such as trip length, destination,
themes, budget details, or essential info). It is also "incorrect" if it adds irrelevant or
contradictory details.

    [BEGIN DATA]
    ************
    [Query]: {{input}}
    ************
    [Answer]: {{output}}
    ************
    [END DATA]

Explain step-by-step how you determined your judgment. Then provide a final LABEL:
- Use "correct" if the answer follows the query accurately and fully.
- Use "incorrect" if it deviates from the query or omits required information.

Your final output must be only one word: "correct" or "incorrect".
"""

In [61]:
nest_asyncio.apply() 
px.launch_app(port=6006)

Existing running Phoenix instance detected! Shutting it down and starting a new instance...


❗️ The launch_app `port` parameter is deprecated and will be removed in a future release. Use the `PHOENIX_PORT` environment variable instead.
🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix


In [89]:
from phoenix.evals import create_classifier
from phoenix.evals.llm import LLM
relevancy_evaluator = create_classifier(
    name="ANSWER RELEVANCE",
    llm=llm,
    prompt_template=ANSWER_RELEVANCE_PROMPT_TEMPLATE,
    choices={"correct": 1.0, "incorrect": 0.0},
)

In [90]:
BUDGET_CONSISTENCY_PROMPT_TEMPLATE = """
You will be given a travel-planning query and an itinerary answer. Your task is to determine whether
the answer provides a consistent and mathematically coherent budget. An answer is "incorrect" if:
- The summed minimum costs of all listed budget categories exceed the stated minimum total estimate.
- The summed maximum costs of all listed budget categories exceed the stated maximum total estimate.
- The total estimate claims a range that cannot be derived from (or contradicted by) the itemized ranges.
- The answer lists budget items but provides a total that is not numerically aligned with them.
- The answer contradicts itself regarding pricing or cost ranges.

    [BEGIN DATA]
    ************
    [Query]: {{input}}
    ************
    [Answer]: {{output}}
    ************
    [END DATA]

Explain step-by-step how you evaluated the itemized costs and the final total, including whether
the ranges mathematically match. Then provide a final LABEL:
- Use "correct" if the budget totals are consistent with the itemized values.
- Use "incorrect" if the totals contradict or cannot be derived from the itemized values.

Your final output must be only one word: "correct" or "incorrect".
"""

In [92]:


budget_evaluator = create_classifier(
    name="BUDGET CONSISTENCY",
    llm=llm,
    prompt_template=BUDGET_CONSISTENCY_PROMPT_TEMPLATE,
    choices={"correct": 1.0, "incorrect": 0.0},
)

In [93]:
def agent_task(input):
    query = input["input"]
    response = trip_agent.run(query, stream=False)
    return response.content

In [94]:
from phoenix.client.experiments import run_experiment

experiment = run_experiment(
    dataset=dataset,
    task=agent_task,
    experiment_name="inital run",
    evaluators=[relevancy_evaluator, budget_evaluator],
)

I0000 00:00:1773843471.196070  224841 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


🧪 Experiment started.
📺 View dataset experiments: http://127.0.0.1:6006/datasets/RGF0YXNldDox/experiments
🔗 View this experiment: http://127.0.0.1:6006/datasets/RGF0YXNldDox/compare?experimentId=RXhwZXJpbWVudDox


running tasks |          | 0/10 (0.0%) | ⏳ 00:00<? | ?it/s

the json <Response [200 OK]>
data {'query': 'Italy travel essentials weather besttime top attraction etiquette', 'follow_up_questions': None, 'answer': 'Best time to visit Italy is September and October; top attraction is Rome; etiquette includes removing shoes in homes.', 'images': [], 'results': [{'title': 'Weather in Italy', 'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'Rome', 'region': 'Lazio', 'country': 'Italy', 'lat': 41.9, 'lon': 12.483, 'tz_id': 'Europe/Rome', 'localtime_epoch': 1773843451, 'localtime': '2026-03-18 15:17'}, 'current': {'last_updated_epoch': 1773843300, 'last_updated': '2026-03-18 15:15', 'temp_c': 15.4, 'temp_f': 59.7, 'is_day': 1, 'condition': {'text': 'Patchy light drizzle', 'icon': '//cdn.weatherapi.com/weather/64x64/day/263.png', 'code': 1150}, 'wind_mph': 3.6, 'wind_kph': 5.8, 'wind_degree': 298, 'wind_dir': 'WNW', 'pressure_mb': 1008.0, 'pressure_in': 29.77, 'precip_mm': 0.26, 'precip_in': 0.01, 'humidity': 48, 'cloud': 25, 'fe

running tasks |█         | 1/10 (10.0%) | ⏳ 00:16<02:28 | 16.50s/it

...

running experiment evaluations |██████    | 12/20 (60.0%) | ⏳ 00:00<00:00 | 35.54it/s

Traceback (most recent call last):
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/phoenix/client/resources/experiments/__init__.py", line 1911, in _run_single_evaluation_sync
    result = evaluator.evaluate(
             ^^^^^^^^^^^^^^^^^^^
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/phoenix/client/resources/experiments/evaluators.py", line 253, in evaluate
    scores = evaluator.evaluate(kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/phoenix/evals/evaluators.py", line 561, in evaluate
    return super().evaluate(eval_input=eval_input, input_mapping=input_mapping)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/phoenix/evals/evaluators.py", line 413, in evaluate
    return 

running experiment evaluations |████████  | 16/20 (80.0%) | ⏳ 00:00<00:00 | 32.03it/s

Traceback (most recent call last):
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/phoenix/client/resources/experiments/__init__.py", line 1911, in _run_single_evaluation_sync
    result = evaluator.evaluate(
             ^^^^^^^^^^^^^^^^^^^
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/phoenix/client/resources/experiments/evaluators.py", line 253, in evaluate
    scores = evaluator.evaluate(kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/phoenix/evals/evaluators.py", line 561, in evaluate
    return super().evaluate(eval_input=eval_input, input_mapping=input_mapping)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/asangtriratnaingle/Desktop/Codes_/embeddings/venv/lib/python3.12/site-packages/phoenix/evals/evaluators.py", line 413, in evaluate
    return 

running experiment evaluations |██████████| 20/20 (100.0%) | ⏳ 2:11:38<00:00 | 394.92s/it


In [62]:
for q in queries:
    print('Q',q)
    print('A',chain.invoke({'question':q}))
    print() 

Q What is AI?
A content='**Artificial Intelligence (AI)** is a broad field of computer science that focuses on creating intelligent machines that can perform tasks that typically require human intelligence. AI involves the development of algorithms, statistical models, and computer programs that enable machines to:\n\n1. **Learn**: from data, experiences, and environments.\n2. **Reason**: using logic, rules, and decision-making processes.\n3. **Problem-solve**: by identifying and solving complex problems.\n4. **Perceive**: through sensors, cameras, and other input devices.\n5. **Interact**: with humans and other machines through natural language, gestures, or other interfaces.\n\nAI systems can be categorized into several types, including:\n\n1. **Narrow or Weak AI**: designed to perform a specific task, such as image recognition, speech recognition, or playing chess.\n2. **General or Strong AI**: capable of performing any intellectual task that a human can, such as reasoning, problem-